# 4d. Forced response figures: probability ratios

The figures for the forced response, drawn from what notebook 03 saved, so nothing slow runs here. Notebook 03 explains each method, step by step. The figures are in four notebooks, so that none grows too big to open; each opens only the saved results it uses, so they can be run in any order after notebook 03:

| Notebook | Sections |
|---|---|
| 04a | 1. Mean change; 2. Signal to noise (and a check of every model's headline numbers) |
| 04b | 3. Quantile response; 4. Change in width, and which tail |
| 04c | 5. The whole distribution; 6. Additivity; 7. Zonal means and the sea-ice edge |
| 04d | 8. Probability ratios; 9. The shift-only ratio |

The sections keep one numbering across the four.

Every section shows its result in the same four figures, with the four seasons across every map:

1. **What goes in.** The quantities the result is made from, then the result, one row each: one model and experiment (`MODEL`, `EXPERIMENT`).
2. **One model.** The parts and the result for every experiment of `MODEL`.
3. **Every model.** The result for every model (rows), one figure per experiment, dotted where it is not significant.
4. **How many models.** The multi-model median of the result (top row), then at each grid point how many models have a significant increase and how many a significant decrease; one figure per experiment.

This notebook:

| Section | Result | Test |
|---|---|---|
| 8. Probability ratios | how much more often a season beyond hist-nat's 95th (warm) or 5th (cold) percentile | bootstrap of members |
| 9. The shift-only ratio | the ratio a shift of the mean alone would give, and what the width and shape add | bootstrap of members |

**By SAM phase.** Sections 8 and 9 end with the result in the seasons of each phase of the Southern Annular Mode alone (negative, neutral and positive, cut at hist-nat's terciles of the index: notebook 03, *SAM Phases*), the phases as rows.

**Reading the figures.** A quantity has the same colour scale in every figure (set once, under *Colour scales*). Every change is red for an increase and blue for a decrease, and dots mark where a change is not significant. The probability ratios are the exception: orange where a season has become more likely, purple where less (section 8). In the counts, the colour changes where more than half the models agree: reds turn brown (an increase) and blues purple (a decrease). The paper figures, which put the models together, are in notebook 06. The cells before each figure pick out its data and show them. Each figure is then drawn layer by layer with `extant/plots/antarctica.py`: `map_figure_size` and `map_axes` make the figure, its maps and colour bars, then each layer (a field, its dots) is drawn on those axes by its own call, a row of a different kind being a slice of them (`axes[2]`).

Names end in their type: `_tree` (DataTree), `_ds` (Dataset), `_da` (DataArray), `_arr` (numpy array), `_df` (DataFrame). Each saved result has the same name as in notebook 03. Until 14 October 2026 the four were one notebook, `04_forced_response_figures.ipynb` (in the git history); the version before that, from 7 October, is in `archive/forced_response_figures_2026-10-07.ipynb`.

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

import numpy as np
import pandas as pd
import xarray as xr

from extant import config, sam
from extant import significance as sig
from extant.logger import setup_logging
from extant.plots import antarctica, charts, forced_response as fr, paper_panels

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)
pd.set_option('display.max_rows', 200)   # every model, experiment and season in one table
#(t): Every figure's text at the notebook's sizes and colours (fr.NOTEBOOK), as in every notebook
plt.rcParams.update(fr.NOTEBOOK.rc)

In [ ]:
%matplotlib inline

## Settings

In [ ]:
VARIABLE = 'tas'

WINDOW = config.WINDOW

#(t): The experiments, in the order of every figure, and the seasons, the columns of every map
EXPERIMENTS = ['hist-GHG', 'hist-aer', 'hist-totalO3', 'historical']
SEASONS = ['DJF', 'MAM', 'JJA', 'SON']

#(t): The model and experiment of the "what goes in" and "one model" figures
MODEL = config.PLOT_SEL['model']
EXPERIMENT = config.PLOT_SEL['experiment']

#(t): The probability ratios: of seasons beyond hist-nat's 95th (warm) or 5th (cold) percentile, which hist-nat has a
#(t): CHANCE of; the tails, in the order of the figures; their percentiles; and the region of 9.3
#(t): (observed_change.REGIONS)
CHANCE = 0.05
TAILS = ['warm', 'cold']
PERCENTILE_LABELS = {'warm': f'{100 * (1 - CHANCE):g}th percentile', 'cold': f'{100 * CHANCE:g}th percentile'}
PR_REGION = 'Antarctica (90–65°S)'

#(t): The SAM phases of 8.5 and 9.4, in the order of their rows, and how the rows are labelled
PHASES = list(sam.PHASES)
PHASE_LABELS = {phase: f'SAM\n{phase}' for phase in PHASES}

#(t): The share of the models that must have a significant change of one sign for a multi-model change to be robust
ROBUST = config.SIGNIFICANT_THRESHOLD


In [ ]:
#(t): Where notebook 03 saved its results: JASMIN's scratch
SCRATCH_DIR = Path('/work/scratch-pw4/aborow/ExtAnt')
SCRATCH_DIR

### Colour scales

One scale per quantity, used by every figure that shows it (here and in notebooks 05 and 06), so the same colour always means the same value. Each is a `fr.Scale`: the contour levels, the colormap and the colour-bar label. Their ranges, per variable, are in `fr.RANGES`; notebook 06 (*Check the Colour Ranges*) says how much of each field falls beyond them.

In [ ]:
scales = fr.colour_scales(VARIABLE)

#(t): The probability ratio (its log2, the ticks labelled with the ratios, as in notebook 07) and the chance of an
#(t): event (sections 8 and 9)
ratio_scale = paper_panels.ratio_scale('Probability ratio')
chance_scale = fr.Scale(np.round(np.arange(0, 1.01, 0.1), 1), 'Reds', 'Chance of the event (share of the seasons)')

In [ ]:
#(t): The legend entries of the maps: the dots where a ratio is not significant, or where the models don't agree
not_significant_handle = plt.Line2D([], [], color='k', marker='.', linestyle='none',
                                    label='Not significant (the 95% range includes 1)')
not_robust_handle = plt.Line2D([], [], color='k', marker='.', linestyle='none',
                               label='Not robust (fewer than two-thirds\nof the models significant with one sign)')

In [ ]:
#(t): What each one covers
pd.DataFrame({name: {'from': scale.levels[0], 'to': scale.levels[-1], 'colormap': getattr(scale.cmap, 'name', scale.cmap),
                     'label': scale.label}
              for name, scale in vars(scales).items()}).T

## Open the Saved Results

Run **one** of the next two sections. Both leave `RESULTS_DIR`, the folder notebook 03's results are opened from.

### Sample data

In [ ]:
#(t): The sample's results
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE / 'sample'
RESULTS_DIR

### Full data

In [ ]:
#(t): The full data's results
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE / 'full'
RESULTS_DIR

### Saved results

Everything sections 8 and 9 use, from `RESULTS_DIR`, one store per cell, loaded into memory.

In [ ]:
#(t): The probability ratios
pr_ds = xr.open_zarr(RESULTS_DIR / 'probability_ratio.zarr').load()
pr_ds

In [ ]:
#(t): The shift-only ratios
pr_shift_only_ds = xr.open_zarr(RESULTS_DIR / 'probability_ratio_shift_only.zarr').load()
pr_shift_only_ds

In [ ]:
#(t): The probability ratios of the regional means
regional_pr_ds = xr.open_zarr(RESULTS_DIR / 'probability_ratio_regional.zarr').load()
regional_pr_ds

In [ ]:
#(t): The shift-only ratios of the regional means
regional_shift_only_ds = xr.open_zarr(RESULTS_DIR / 'probability_ratio_shift_only_regional.zarr').load()
regional_shift_only_ds

In [ ]:
#(t): The probability ratios in each SAM phase
sam_pr_ds = xr.open_zarr(RESULTS_DIR / 'sam_probability_ratio.zarr').load()
sam_pr_ds

In [ ]:
#(t): The shift-only ratios in each SAM phase
sam_pr_shift_only_ds = xr.open_zarr(RESULTS_DIR / 'sam_probability_ratio_shift_only.zarr').load()
sam_pr_shift_only_ds

In [ ]:
#(t): The number of models: the top of the colour bar of every count figure
N_MODELS = pr_ds.sizes['model']
N_MODELS

## 8. Probability Ratios

How much more often each experiment has a season warmer than hist-nat's 95th percentile (a warm season), or colder than its 5th (a cold one), over the final 21 years; both percentiles are hist-nat's own, over the same years (notebook 03, section 10):

$$
\mathrm{PR} = \frac{P_{\mathrm{exp}}}{P_{\mathrm{nat}}},
$$

the share of the experiment's seasons beyond the percentile over hist-nat's share (about 5%, by construction). PR = 2 doubles the chance and PR = 1/2 halves it. The maps show $\log_2 \mathrm{PR}$, labelled with the ratios: orange is more likely, purple less likely. A ratio is significant where the 95% range of its bootstrap (whole members resampled) is clear of 1.

### 8.1 What goes in

hist-nat's percentile, the chance of a season beyond it in hist-nat and in `EXPERIMENT`, and their ratio, for `MODEL`: one figure for warm seasons and one for cold.

In [ ]:
#(t): MODEL's probability ratios, every experiment, season and tail
model_pr_ds = pr_ds.sel(model=MODEL, experiment=EXPERIMENTS, season=SEASONS, chance=CHANCE)
model_pr_ds

In [ ]:
#(t): EXPERIMENT's
experiment_pr_ds = model_pr_ds.sel(experiment=EXPERIMENT)
experiment_pr_ds

In [ ]:
#(t): The ratio as log2, as the maps show it (a ratio of 0 kept finite)
experiment_log_ratio_da = np.log2(experiment_pr_ds['pr'].clip(min=2.0 ** -10))
experiment_log_ratio_da

In [ ]:
#(t): Not significant: the bootstrap's 95% range includes 1 (False where there is no data)
experiment_pr_not_significant_da = ~experiment_pr_ds['pr_significant'] & experiment_pr_ds['pr'].notnull()
experiment_pr_not_significant_da

In [ ]:
#(t): What goes in, one figure per tail: hist-nat's threshold, the chance of passing it in hist-nat and in EXPERIMENT
#(t): (one colour bar), and the probability ratio, dotted where not significant
for tail in TAILS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=4, n_cols=len(SEASONS), n_colorbars=3))
    axes, cbar_axes = antarctica.map_axes(fig, n_rows=4, n_cols=len(SEASONS), n_colorbars=3)

    antarctica.heatmaps_of_antarctica(experiment_pr_ds['threshold'].sel(tail=tail), None, 'season', axes=axes[0],
                                      cax=cbar_axes[0], **scales.temperature.colours())
    antarctica.heatmaps_of_antarctica(experiment_pr_ds['p_counterfactual'].sel(tail=tail), None, 'season', axes=axes[1],
                                      cax=cbar_axes[1], **chance_scale.colours())
    antarctica.heatmaps_of_antarctica(experiment_pr_ds['p_factual'].sel(tail=tail), None, 'season', axes=axes[2],
                                      **chance_scale.colours())
    antarctica.heatmaps_of_antarctica(experiment_log_ratio_da.sel(tail=tail), None, 'season', axes=axes[3],
                                      cax=cbar_axes[2], **ratio_scale.colours())
    antarctica.not_significant_dots_grid(experiment_pr_not_significant_da.sel(tail=tail), None, 'season', axes=axes[3])

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, [f'hist-nat\n{PERCENTILE_LABELS[tail]}', 'Chance\nhist-nat', f'Chance\n{EXPERIMENT}',
                                 'Probability\nratio'])
    fig.legend(handles=[not_significant_handle], loc='lower right', fontsize=8)
    fig.suptitle(f'What goes into the probability ratio: {MODEL}, {EXPERIMENT}, {tail} seasons')

### 8.2 One model

The probability ratio for every experiment of `MODEL`, dotted where it is not significant: one figure for warm seasons and one for cold.

In [ ]:
#(t): MODEL's ratios as log2, every experiment
model_log_ratio_da = np.log2(model_pr_ds['pr'].clip(min=2.0 ** -10))
model_log_ratio_da

In [ ]:
#(t): Not significant, where there is data
model_pr_not_significant_da = ~model_pr_ds['pr_significant'] & model_pr_ds['pr'].notnull()
model_pr_not_significant_da

In [ ]:
#(t): MODEL's probability ratio in every experiment (rows), one figure per tail, dotted where not significant
rows = EXPERIMENTS
for tail in TAILS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1))
    axes, cbar_axes = antarctica.map_axes(fig, n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1)

    antarctica.heatmaps_of_antarctica(model_log_ratio_da.sel(tail=tail), 'experiment', 'season', axes=axes, cax=cbar_axes[0],
                                      **ratio_scale.colours())
    antarctica.not_significant_dots_grid(model_pr_not_significant_da.sel(tail=tail), 'experiment', 'season', axes=axes)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, rows)
    fig.legend(handles=[not_significant_handle], loc='lower right', fontsize=8)
    fig.suptitle(f'{MODEL}: probability ratio of {tail} seasons, final {WINDOW} years against hist-nat')

### 8.3 Every model

One figure per experiment: every model down, every season across. Warm seasons, then cold.

In [ ]:
#(t): Every model's probability ratios
all_pr_ds = pr_ds.sel(experiment=EXPERIMENTS, season=SEASONS, chance=CHANCE)
all_pr_ds

In [ ]:
#(t): as log2
all_log_ratio_da = np.log2(all_pr_ds['pr'].clip(min=2.0 ** -10))
all_log_ratio_da

In [ ]:
#(t): Significant: the bootstrap's 95% range is clear of 1
all_pr_significant_da = all_pr_ds['pr_significant']
all_pr_significant_da

In [ ]:
#(t): Not significant, where there is data
all_pr_not_significant_da = ~all_pr_significant_da & all_pr_ds['pr'].notnull()
all_pr_not_significant_da

In [ ]:
#(t): Every model's probability ratio of warm seasons (rows), one figure per experiment, dotted where not significant
for experiment in EXPERIMENTS:
    rows = all_log_ratio_da.sel(tail='warm', experiment=experiment)['model'].values
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1))
    axes, cbar_axes = antarctica.map_axes(fig, n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1)

    antarctica.heatmaps_of_antarctica(all_log_ratio_da.sel(tail='warm', experiment=experiment), 'model', 'season', axes=axes, cax=cbar_axes[0],
                                      **ratio_scale.colours())
    antarctica.not_significant_dots_grid(all_pr_not_significant_da.sel(tail='warm', experiment=experiment), 'model', 'season', axes=axes)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, rows)
    fig.legend(handles=[not_significant_handle], loc='lower right', fontsize=8)
    fig.suptitle(f'Probability ratio of warm seasons, every model: {experiment}')

In [ ]:
#(t): Every model's probability ratio of cold seasons (rows), one figure per experiment, dotted where not significant
for experiment in EXPERIMENTS:
    rows = all_log_ratio_da.sel(tail='cold', experiment=experiment)['model'].values
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1))
    axes, cbar_axes = antarctica.map_axes(fig, n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1)

    antarctica.heatmaps_of_antarctica(all_log_ratio_da.sel(tail='cold', experiment=experiment), 'model', 'season', axes=axes, cax=cbar_axes[0],
                                      **ratio_scale.colours())
    antarctica.not_significant_dots_grid(all_pr_not_significant_da.sel(tail='cold', experiment=experiment), 'model', 'season', axes=axes)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, rows)
    fig.legend(handles=[not_significant_handle], loc='lower right', fontsize=8)
    fig.suptitle(f'Probability ratio of cold seasons, every model: {experiment}')

### 8.4 How many models

The multi-model median ratio (top row), then at each grid point how many models have a ratio significantly above 1 (more likely) and how many significantly below 1 (less likely). One figure per experiment; warm seasons, then cold.

In [ ]:
#(t): The multi-model median of the log2 ratios
median_log_ratio_da = all_log_ratio_da.median('model')
median_log_ratio_da

In [ ]:
#(t): How many models have a ratio significantly above 1, and how many significantly below
pr_counts_da = sig.significant_counts(all_log_ratio_da, all_pr_significant_da)
pr_counts_da

In [ ]:
#(t): The ratios significantly below 1 as negative counts, so that one colour bar (blues for less likely, reds for
#(t): more likely) serves both rows of counts
signed_pr_counts_da = xr.concat([pr_counts_da.sel(direction=['increase']), -pr_counts_da.sel(direction=['decrease'])],
                                dim='direction')
signed_pr_counts_da

In [ ]:
#(t): Their colours: blues and reds up to half the models, purples and browns beyond, white for none
pr_count_colours = fr.count_scale(N_MODELS, 'Number of models with a ratio significantly below 1 (blue) or above 1 '
                                            '(red)').colours()

In [ ]:
#(t): The multi-model median of the ratio of warm seasons (top row), then how many models have one significantly
#(t): above and below 1, one figure per experiment
for experiment in EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=3, n_cols=len(SEASONS), n_colorbars=2))
    axes, cbar_axes = antarctica.map_axes(fig, n_rows=3, n_cols=len(SEASONS), n_colorbars=2)

    antarctica.heatmaps_of_antarctica(median_log_ratio_da.sel(tail='warm', experiment=experiment), None, 'season',
                                      axes=axes[0], cax=cbar_axes[0], **ratio_scale.colours())
    antarctica.heatmaps_of_antarctica(signed_pr_counts_da.sel(tail='warm', experiment=experiment), 'direction',
                                      'season', axes=axes[1:], cax=cbar_axes[1], **pr_count_colours)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, ['Multi-model\nmedian', 'More\nlikely', 'Less\nlikely'])
    fig.suptitle(f'How many models have a significant probability ratio: warm seasons, {experiment}')

In [ ]:
#(t): The multi-model median of the ratio of cold seasons (top row), then how many models have one significantly
#(t): above and below 1, one figure per experiment
for experiment in EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=3, n_cols=len(SEASONS), n_colorbars=2))
    axes, cbar_axes = antarctica.map_axes(fig, n_rows=3, n_cols=len(SEASONS), n_colorbars=2)

    antarctica.heatmaps_of_antarctica(median_log_ratio_da.sel(tail='cold', experiment=experiment), None, 'season',
                                      axes=axes[0], cax=cbar_axes[0], **ratio_scale.colours())
    antarctica.heatmaps_of_antarctica(signed_pr_counts_da.sel(tail='cold', experiment=experiment), 'direction',
                                      'season', axes=axes[1:], cax=cbar_axes[1], **pr_count_colours)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, ['Multi-model\nmedian', 'More\nlikely', 'Less\nlikely'])
    fig.suptitle(f'How many models have a significant probability ratio: cold seasons, {experiment}')

### 8.5 By SAM phase

The probability ratio in the seasons of each SAM phase alone (notebook 03, section 10, *By SAM phase*): the same events (beyond hist-nat's 95th or 5th percentile over all its seasons), their chance among the seasons of one phase, against hist-nat's chance in the same phase. For `MODEL`, then the multi-model median, dotted where it is not robust, then how many models; warm seasons, then cold.

In [ ]:
#(t): MODEL's ratios in each phase, every experiment and season
model_phase_pr_ds = sam_pr_ds.sel(model=MODEL, experiment=EXPERIMENTS, season=SEASONS, chance=CHANCE)
model_phase_pr_ds

In [ ]:
#(t): as log2
model_phase_log_ratio_da = np.log2(model_phase_pr_ds['pr'].clip(min=2.0 ** -10))
model_phase_log_ratio_da

In [ ]:
#(t): Not significant, where there is data
model_phase_pr_not_significant_da = ~model_phase_pr_ds['pr_significant'] & model_phase_pr_ds['pr'].notnull()
model_phase_pr_not_significant_da

In [ ]:
#(t): MODEL's probability ratio of warm seasons in each SAM phase (rows), one figure per experiment, dotted where not
#(t): significant
rows = model_phase_log_ratio_da['phase'].values
for experiment in EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1))
    axes, cbar_axes = antarctica.map_axes(fig, n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1)

    antarctica.heatmaps_of_antarctica(model_phase_log_ratio_da.sel(tail='warm', experiment=experiment), 'phase', 'season', axes=axes, cax=cbar_axes[0],
                                      **ratio_scale.colours())
    antarctica.not_significant_dots_grid(model_phase_pr_not_significant_da.sel(tail='warm', experiment=experiment), 'phase', 'season', axes=axes)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, [PHASE_LABELS[phase] for phase in rows])
    fig.legend(handles=[not_significant_handle], loc='lower right', fontsize=8)
    fig.suptitle(f'{MODEL}: probability ratio of warm seasons in each SAM phase: {experiment}')

In [ ]:
#(t): MODEL's probability ratio of cold seasons in each SAM phase (rows), one figure per experiment, dotted where not
#(t): significant
rows = model_phase_log_ratio_da['phase'].values
for experiment in EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1))
    axes, cbar_axes = antarctica.map_axes(fig, n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1)

    antarctica.heatmaps_of_antarctica(model_phase_log_ratio_da.sel(tail='cold', experiment=experiment), 'phase', 'season', axes=axes, cax=cbar_axes[0],
                                      **ratio_scale.colours())
    antarctica.not_significant_dots_grid(model_phase_pr_not_significant_da.sel(tail='cold', experiment=experiment), 'phase', 'season', axes=axes)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, [PHASE_LABELS[phase] for phase in rows])
    fig.legend(handles=[not_significant_handle], loc='lower right', fontsize=8)
    fig.suptitle(f'{MODEL}: probability ratio of cold seasons in each SAM phase: {experiment}')

In [ ]:
#(t): Every model's ratios in each phase
all_phase_pr_ds = sam_pr_ds.sel(experiment=EXPERIMENTS, season=SEASONS, chance=CHANCE)
all_phase_pr_ds

In [ ]:
#(t): as log2
all_phase_log_ratio_da = np.log2(all_phase_pr_ds['pr'].clip(min=2.0 ** -10))
all_phase_log_ratio_da

In [ ]:
#(t): The multi-model median
median_phase_log_ratio_da = all_phase_log_ratio_da.median('model')
median_phase_log_ratio_da

In [ ]:
#(t): Where the models agree: +1 (-1) where at least ROBUST of them have a ratio significantly above (below) 1
phase_pr_robust_da = sig.robust_sign(all_phase_log_ratio_da, all_phase_pr_ds['pr_significant'], threshold=ROBUST)
phase_pr_robust_da

In [ ]:
#(t): Not robust, where there is data
phase_pr_not_robust_da = phase_pr_robust_da == 0
phase_pr_not_robust_da

In [ ]:
#(t): The multi-model median ratio of warm seasons in each SAM phase (rows), one figure per experiment, dotted where
#(t): not robust
rows = median_phase_log_ratio_da['phase'].values
for experiment in EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1))
    axes, cbar_axes = antarctica.map_axes(fig, n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1)

    antarctica.heatmaps_of_antarctica(median_phase_log_ratio_da.sel(tail='warm', experiment=experiment), 'phase', 'season', axes=axes, cax=cbar_axes[0],
                                      **ratio_scale.colours())
    antarctica.not_significant_dots_grid(phase_pr_not_robust_da.sel(tail='warm', experiment=experiment), 'phase', 'season', axes=axes)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, [PHASE_LABELS[phase] for phase in rows])
    fig.legend(handles=[not_robust_handle], loc='lower right', fontsize=8)
    fig.suptitle(f'Probability ratio of warm seasons in each SAM phase, multi-model median: {experiment}')

In [ ]:
#(t): The multi-model median ratio of cold seasons in each SAM phase (rows), one figure per experiment, dotted where
#(t): not robust
rows = median_phase_log_ratio_da['phase'].values
for experiment in EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1))
    axes, cbar_axes = antarctica.map_axes(fig, n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1)

    antarctica.heatmaps_of_antarctica(median_phase_log_ratio_da.sel(tail='cold', experiment=experiment), 'phase', 'season', axes=axes, cax=cbar_axes[0],
                                      **ratio_scale.colours())
    antarctica.not_significant_dots_grid(phase_pr_not_robust_da.sel(tail='cold', experiment=experiment), 'phase', 'season', axes=axes)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, [PHASE_LABELS[phase] for phase in rows])
    fig.legend(handles=[not_robust_handle], loc='lower right', fontsize=8)
    fig.suptitle(f'Probability ratio of cold seasons in each SAM phase, multi-model median: {experiment}')

In [ ]:
#(t): How many models have a ratio significantly above 1, and how many significantly below, in each phase
phase_pr_counts_da = sig.significant_counts(all_phase_log_ratio_da, all_phase_pr_ds['pr_significant'])
phase_pr_counts_da

In [ ]:
#(t): The ratios significantly below 1 as negative counts, as in 8.4
signed_phase_pr_counts_da = xr.concat([phase_pr_counts_da.sel(direction=['increase']),
                                       -phase_pr_counts_da.sel(direction=['decrease'])], dim='direction')
signed_phase_pr_counts_da

In [ ]:
#(t): How many models have a ratio of warm seasons significantly above and below 1 in each SAM phase: two rows per
#(t): phase, one colour bar
phases = signed_phase_pr_counts_da['phase'].values
for experiment in EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=2 * len(phases), n_cols=len(SEASONS), n_colorbars=1))
    axes, cbar_axes = antarctica.map_axes(fig, n_rows=2 * len(phases), n_cols=len(SEASONS), n_colorbars=1)

    #(c): Each phase's two rows, more likely then less likely; the colour bar drawn once, with the first phase
    for k, phase in enumerate(phases):
        antarctica.heatmaps_of_antarctica(signed_phase_pr_counts_da.sel(tail='warm', experiment=experiment, phase=phase),
                                          'direction', 'season', axes=axes[2 * k:2 * k + 2],
                                          cax=cbar_axes[0] if k == 0 else None, **pr_count_colours)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, [f'SAM {phase}\n{likely}' for phase in phases for likely in ['more likely', 'less likely']])
    fig.suptitle(f'How many models have a significant probability ratio in each SAM phase: warm seasons, {experiment}')

In [ ]:
#(t): How many models have a ratio of cold seasons significantly above and below 1 in each SAM phase: two rows per
#(t): phase, one colour bar
phases = signed_phase_pr_counts_da['phase'].values
for experiment in EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=2 * len(phases), n_cols=len(SEASONS), n_colorbars=1))
    axes, cbar_axes = antarctica.map_axes(fig, n_rows=2 * len(phases), n_cols=len(SEASONS), n_colorbars=1)

    #(c): Each phase's two rows, more likely then less likely; the colour bar drawn once, with the first phase
    for k, phase in enumerate(phases):
        antarctica.heatmaps_of_antarctica(signed_phase_pr_counts_da.sel(tail='cold', experiment=experiment, phase=phase),
                                          'direction', 'season', axes=axes[2 * k:2 * k + 2],
                                          cax=cbar_axes[0] if k == 0 else None, **pr_count_colours)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, [f'SAM {phase}\n{likely}' for phase in phases for likely in ['more likely', 'less likely']])
    fig.suptitle(f'How many models have a significant probability ratio in each SAM phase: cold seasons, {experiment}')

## 9. Shift-Only Ratio

The ratio a shift of the mean alone would give: hist-nat's seasons, every one moved by the experiment's change in the mean (final 21 years, experiment minus hist-nat), counted against the same percentiles (notebook 03, section 11). Where the probability ratio is bigger than its shift-only ratio, the change in width or shape has made the event more likely still; where it is smaller, a narrowing has held it back.

For example: hist-nat goes beyond its warm percentile 5% of the time and hist-GHG 15%, so PR = 3. hist-nat's seasons moved up by hist-GHG's warming go beyond it 12% of the time, a shift-only ratio of 2.4: the warming gives most of the 3, and the widening the rest. The *ratio over shift-only* below is PR divided by the shift-only ratio (here 3 / 2.4 = 1.25): what the width and shape add.

### 9.1 What goes in

The change in the mean, the shift-only ratio, the probability ratio (section 8) and the one over the other, for `MODEL` and `EXPERIMENT`: one figure for warm seasons and one for cold.

In [ ]:
#(t): MODEL's shift-only ratios, every experiment, season and tail
model_shift_only_ds = pr_shift_only_ds.sel(model=MODEL, experiment=EXPERIMENTS, season=SEASONS, chance=CHANCE)
model_shift_only_ds

In [ ]:
#(t): EXPERIMENT's
experiment_shift_only_ds = model_shift_only_ds.sel(experiment=EXPERIMENT)
experiment_shift_only_ds

In [ ]:
#(t): The shift-only ratio as log2
experiment_log_shift_only_da = np.log2(experiment_shift_only_ds['pr_shift_only'].clip(min=2.0 ** -10))
experiment_log_shift_only_da

In [ ]:
#(t): The ratio over the shift-only ratio, as log2: what the width and shape add
experiment_log_gap_da = experiment_log_ratio_da - experiment_log_shift_only_da
experiment_log_gap_da

In [ ]:
#(t): What goes in, one figure per tail: the shift of the mean, then the shift-only ratio, the probability ratio and
#(t): the one over the other (one colour bar)
for tail in TAILS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=4, n_cols=len(SEASONS), n_colorbars=2))
    axes, cbar_axes = antarctica.map_axes(fig, n_rows=4, n_cols=len(SEASONS), n_colorbars=2)

    antarctica.heatmaps_of_antarctica(experiment_shift_only_ds['shift'], None, 'season', axes=axes[0], cax=cbar_axes[0],
                                      **scales.change.colours())
    antarctica.heatmaps_of_antarctica(experiment_log_shift_only_da.sel(tail=tail), None, 'season', axes=axes[1],
                                      cax=cbar_axes[1], **ratio_scale.colours())
    antarctica.heatmaps_of_antarctica(experiment_log_ratio_da.sel(tail=tail), None, 'season', axes=axes[2],
                                      **ratio_scale.colours())
    antarctica.heatmaps_of_antarctica(experiment_log_gap_da.sel(tail=tail), None, 'season', axes=axes[3],
                                      **ratio_scale.colours())

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, [f'Shift of the mean\n{EXPERIMENT} − hist-nat', 'Shift-only\nratio', 'Probability\nratio',
                                 'Ratio over\nshift-only'])
    fig.suptitle(f'What goes into the shift-only ratio: {MODEL}, {EXPERIMENT}, {tail} seasons')

### 9.2 One model

For every experiment of `MODEL`: the shift-only ratio, and the probability ratio over it (what the width and shape add). One figure each, warm seasons and then cold.

In [ ]:
#(t): MODEL's shift-only ratios as log2, every experiment
model_log_shift_only_da = np.log2(model_shift_only_ds['pr_shift_only'].clip(min=2.0 ** -10))
model_log_shift_only_da

In [ ]:
#(t): The ratio over the shift-only ratio, as log2
model_log_gap_da = model_log_ratio_da - model_log_shift_only_da
model_log_gap_da

In [ ]:
#(t): MODEL's shift-only ratio in every experiment (rows), one figure per tail
rows = EXPERIMENTS
for tail in TAILS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1))
    axes, cbar_axes = antarctica.map_axes(fig, n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1)

    antarctica.heatmaps_of_antarctica(model_log_shift_only_da.sel(tail=tail), 'experiment', 'season', axes=axes, cax=cbar_axes[0],
                                      **ratio_scale.colours())

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, rows)
    fig.suptitle(f'{MODEL}: shift-only ratio of {tail} seasons, final {WINDOW} years against hist-nat')

In [ ]:
#(t): MODEL's probability ratio over the shift-only ratio in every experiment (rows), one figure per tail
rows = EXPERIMENTS
for tail in TAILS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1))
    axes, cbar_axes = antarctica.map_axes(fig, n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1)

    antarctica.heatmaps_of_antarctica(model_log_gap_da.sel(tail=tail), 'experiment', 'season', axes=axes, cax=cbar_axes[0],
                                      **ratio_scale.colours())

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, rows)
    fig.suptitle(f'{MODEL}: probability ratio over the shift-only ratio, {tail} seasons (what the width and shape add)')

### 9.3 Every model, over a region

Every model's probability ratio over `PR_REGION` (filled markers, with the 95% range of its bootstrap) and its shift-only ratio (open markers), for every experiment and season; bars are the multi-model median. The gap between a filled and an open marker is what the width and shape add.

In [ ]:
#(t): The ratios over PR_REGION, every model
region_pr_ds = regional_pr_ds.sel(region=PR_REGION, chance=CHANCE)
region_pr_ds

In [ ]:
#(t): and the shift-only ratios
region_shift_only_ds = regional_shift_only_ds.sel(region=PR_REGION, chance=CHANCE)
region_shift_only_ds

In [ ]:
#(t): The two side by side
region_ratios_ds = xr.merge([region_pr_ds, region_shift_only_ds])
region_ratios_ds[['pr', 'pr_lower', 'pr_upper', 'pr_shift_only']].to_dataframe().unstack('season').round(2)

In [ ]:
#(t): Probability ratios over PR_REGION: every model (markers, with their bootstrap ranges), the multi-model median
#(t): (bars) and the ratio the shift of the mean alone would give (open markers); the warm and the cold tail down,
#(t): seasons across
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=2, n_cols=len(SEASONS), n_legends=1))
axes, legend_axes = charts.chart_axes(fig, n_rows=2, n_cols=len(SEASONS), n_legends=1, sharex=True, sharey=True)
paper_panels.probability_ratios_grid(region_ratios_ds.sel(tail=['warm', 'cold'], experiment=EXPERIMENTS, season=SEASONS),
                                     'tail', 'season', axes=axes, shift_only=True)
for ax, season in zip(axes[0], SEASONS):
    ax.set_title(season, loc='left')
for ax, tail in zip(axes[:, 0], ['warm', 'cold']):
    ax.set_ylabel(f'{tail}\nProbability ratio')
legend_axes[0].legend(handles=charts.model_legend_handles(region_ratios_ds['model'].values)
                      + [Patch(color='0.6', alpha=0.5, label='Multi-model median'),
                         Line2D([], [], linestyle='none', marker='o', markersize=5, markerfacecolor='white',
                                markeredgecolor='0.35', label='Shift of the mean only')],
                      loc='center', ncols=region_ratios_ds.sizes['model'] + 2, frameon=False)
fig.suptitle(f'Probability ratios over {PR_REGION}: every model, and the shift-only ratio (open markers)', x=0.01,
             ha='left');

### 9.4 By SAM phase

The shift-only ratio in each SAM phase (notebook 03, section 11, *By SAM phase*), and the probability ratio over it: what the width and shape add in each phase. The multi-model medians, one figure per experiment; warm seasons, then cold.

In [ ]:
#(t): Every model's shift-only ratios in each phase
all_phase_shift_only_ds = sam_pr_shift_only_ds.sel(experiment=EXPERIMENTS, season=SEASONS, chance=CHANCE)
all_phase_shift_only_ds

In [ ]:
#(t): as log2
all_phase_log_shift_only_da = np.log2(all_phase_shift_only_ds['pr_shift_only'].clip(min=2.0 ** -10))
all_phase_log_shift_only_da

In [ ]:
#(t): The probability ratio over the shift-only ratio in each phase, as log2 (all_phase_log_ratio_da: 8.5)
all_phase_log_gap_da = all_phase_log_ratio_da - all_phase_log_shift_only_da
all_phase_log_gap_da

In [ ]:
#(t): The multi-model medians
median_phase_log_shift_only_da = all_phase_log_shift_only_da.median('model')
median_phase_log_gap_da = all_phase_log_gap_da.median('model')
median_phase_log_gap_da

In [ ]:
#(t): Shift-only ratio of warm seasons in each SAM phase (rows), multi-model median, one figure per experiment
rows = median_phase_log_shift_only_da['phase'].values
for experiment in EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1))
    axes, cbar_axes = antarctica.map_axes(fig, n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1)

    antarctica.heatmaps_of_antarctica(median_phase_log_shift_only_da.sel(tail='warm', experiment=experiment), 'phase', 'season', axes=axes, cax=cbar_axes[0],
                                      **ratio_scale.colours())

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, [PHASE_LABELS[phase] for phase in rows])
    fig.suptitle(f'Shift-only ratio of warm seasons in each SAM phase, multi-model median: {experiment}')

In [ ]:
#(t): Shift-only ratio of cold seasons in each SAM phase (rows), multi-model median, one figure per experiment
rows = median_phase_log_shift_only_da['phase'].values
for experiment in EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1))
    axes, cbar_axes = antarctica.map_axes(fig, n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1)

    antarctica.heatmaps_of_antarctica(median_phase_log_shift_only_da.sel(tail='cold', experiment=experiment), 'phase', 'season', axes=axes, cax=cbar_axes[0],
                                      **ratio_scale.colours())

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, [PHASE_LABELS[phase] for phase in rows])
    fig.suptitle(f'Shift-only ratio of cold seasons in each SAM phase, multi-model median: {experiment}')

In [ ]:
#(t): Probability ratio over the shift-only ratio, warm seasons in each SAM phase (rows), multi-model median, one figure per experiment
rows = median_phase_log_gap_da['phase'].values
for experiment in EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1))
    axes, cbar_axes = antarctica.map_axes(fig, n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1)

    antarctica.heatmaps_of_antarctica(median_phase_log_gap_da.sel(tail='warm', experiment=experiment), 'phase', 'season', axes=axes, cax=cbar_axes[0],
                                      **ratio_scale.colours())

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, [PHASE_LABELS[phase] for phase in rows])
    fig.suptitle(f'Probability ratio over the shift-only ratio, warm seasons in each SAM phase, multi-model median: {experiment}')

In [ ]:
#(t): Probability ratio over the shift-only ratio, cold seasons in each SAM phase (rows), multi-model median, one figure per experiment
rows = median_phase_log_gap_da['phase'].values
for experiment in EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1))
    axes, cbar_axes = antarctica.map_axes(fig, n_rows=len(rows), n_cols=len(SEASONS), n_colorbars=1)

    antarctica.heatmaps_of_antarctica(median_phase_log_gap_da.sel(tail='cold', experiment=experiment), 'phase', 'season', axes=axes, cax=cbar_axes[0],
                                      **ratio_scale.colours())

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, [PHASE_LABELS[phase] for phase in rows])
    fig.suptitle(f'Probability ratio over the shift-only ratio, cold seasons in each SAM phase, multi-model median: {experiment}')